In [ ]:
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Загружаем датасет (локальный файл — offline)
df = pd.read_csv('../../data/california_housing.csv')
df = df.rename(columns={'MedHouseVal': 'Price'})

df.head()

In [ ]:
X = df.drop(columns=['Price'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)


In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

# Проверка на NaN/inf перед обучением
assert np.isfinite(X_train.to_numpy()).all(), "В X_train есть NaN или inf"
assert np.isfinite(X_test.to_numpy()).all(), "В X_test есть NaN или inf"

model.fit(X_train, y_train)

print("Модель обучена!")


In [ ]:
# На некоторых сборках NumPy/BLAS могут появляться ложные RuntimeWarning при matmul
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=RuntimeWarning, module=r"sklearn\.linear_model\._base")
    y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")


In [ ]:
baseline = y_train.mean()
baseline_pred = [baseline] * len(y_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline MAE: {baseline_mae:.2f}")
print(f"Model MAE: {mae:.2f}")